In [1]:
"""

Problem solved
--------------
    u_t + u_x = 0,   x in [0,1), t in [0,1]
with periodic BC and Gaussian IC
    u(x,0) = exp( - d_per(x, x0)^2 / (2 nu^2) )

"""

import os
import csv
import math
import random
import argparse
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ============================================================
# Reproducibility
# ============================================================
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# Utilities
# ============================================================
def get_device(use_gpu_flag: bool) -> torch.device:
    if use_gpu_flag and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def periodic_distance_torch(x: torch.Tensor, mu: torch.Tensor) -> torch.Tensor:
    """Periodic wrapped distance on [0,1)."""
    return torch.remainder(x - mu + 0.5, 1.0) - 0.5


def periodic_distance_numpy(x: np.ndarray, mu: np.ndarray) -> np.ndarray:
    return np.remainder(x - mu + 0.5, 1.0) - 0.5


def periodic_gaussian_torch(x: torch.Tensor, x0: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    d = periodic_distance_torch(x, x0)
    return torch.exp(-(d ** 2) / (2.0 * nu ** 2))


def periodic_gaussian_numpy(x: np.ndarray, x0: float, nu: float) -> np.ndarray:
    d = periodic_distance_numpy(x, x0)
    return np.exp(-(d ** 2) / (2.0 * nu ** 2))


def fourier_features(t: torch.Tensor, n_freq: int) -> torch.Tensor:
    """Return [t, sin(2πkt), cos(2πkt)] features."""
    feats = [t]
    for k in range(1, n_freq + 1):
        feats.append(torch.sin(2.0 * math.pi * k * t))
        feats.append(torch.cos(2.0 * math.pi * k * t))
    return torch.cat(feats, dim=-1)


# ============================================================
# Model
# ============================================================
class DynamicMetaSPINN(nn.Module):
    def __init__(
        self,
        M: int = 32,
        hidden_task: int = 64,
        hidden_time: int = 96,
        n_fourier: int = 4,
        h_min: float = 0.015,
        h_max: float = 0.25,
        vel_scale: float = 0.25,
    ) -> None:
        super().__init__()
        self.M = M
        self.n_fourier = n_fourier
        self.h_min = h_min
        self.h_max = h_max
        self.vel_scale = vel_scale

        # Task encoder from p=[x0,nu]
        self.task_net = nn.Sequential(
            nn.Linear(2, hidden_task),
            nn.Tanh(),
            nn.Linear(hidden_task, hidden_task),
            nn.Tanh(),
        )

        # Initial basis geometry from task only
        self.init_head = nn.Linear(hidden_task, 4 * M)
        # -> gate, xi0, h0, a0

        # Dynamic update from (time features, task embedding)
        in_dyn = hidden_task + (1 + 2 * n_fourier)
        self.dyn_net = nn.Sequential(
            nn.Linear(in_dyn, hidden_time),
            nn.Tanh(),
            nn.Linear(hidden_time, hidden_time),
            nn.Tanh(),
            nn.Linear(hidden_time, 4 * M),
        )
        # -> delta_xi, delta_h, alpha, vel

    def normalize_p(self, p: torch.Tensor) -> torch.Tensor:
        mean = p.new_tensor([0.5, 0.075])
        std = p.new_tensor([0.3, 0.03])
        return (p - mean) / std

    def basis_params(self, p: torch.Tensor, t: torch.Tensor):
        """
        p: (B,2)
        t: (B,N,1)
        returns alpha, xi, h, gate with shapes (B,N,M)
        """
        B, N, _ = t.shape
        p_norm = self.normalize_p(p)
        emb = self.task_net(p_norm)  # (B,hidden)

        init_out = self.init_head(emb).view(B, 4, self.M)
        gate_logits = init_out[:, 0, :]
        xi0_raw = init_out[:, 1, :]
        h0_raw = init_out[:, 2, :]
        a0_raw = init_out[:, 3, :]

        gate = torch.sigmoid(gate_logits)                        # (B,M)
        xi0 = torch.sigmoid(xi0_raw)                             # (B,M) in [0,1]
        h0 = self.h_min + (self.h_max - self.h_min) * torch.sigmoid(h0_raw)  # (B,M)
        a0 = torch.tanh(a0_raw)                                  # (B,M)

        tf = fourier_features(t.reshape(B * N, 1), self.n_fourier).reshape(B, N, -1)
        emb_rep = emb.unsqueeze(1).expand(-1, N, -1)
        dyn_in = torch.cat([emb_rep, tf], dim=-1)
        dyn_out = self.dyn_net(dyn_in).view(B, N, 4, self.M)

        dxi_raw = dyn_out[:, :, 0, :]
        dh_raw = dyn_out[:, :, 1, :]
        alpha_raw = dyn_out[:, :, 2, :]
        vel_raw = dyn_out[:, :, 3, :]

        # Time-dependent center motion without prescribing characteristics.
        # The network learns center trajectories via a combination of direct shift and velocity-like term.
        xi = torch.remainder(
            xi0.unsqueeze(1)
            + 0.35 * torch.tanh(dxi_raw)
            + self.vel_scale * t * torch.tanh(vel_raw),
            1.0,
        )

        h = torch.clamp(
            h0.unsqueeze(1) * (1.0 + 0.35 * torch.tanh(dh_raw)),
            min=self.h_min,
            max=self.h_max,
        )

        alpha = a0.unsqueeze(1) + 1.0 * torch.tanh(alpha_raw)
        gate_full = gate.unsqueeze(1).expand(-1, N, -1)
        return alpha, xi, h, gate_full

    def forward(self, p: torch.Tensor, xt: torch.Tensor):
        """
        p:  (B,2)     task parameters [x0, nu]
        xt: (B,N,2)   query points [x, t]
        returns u: (B,N), aux tuple
        """
        x = xt[:, :, 0:1]  # (B,N,1)
        t = xt[:, :, 1:2]  # (B,N,1)
        alpha, xi, h, gate = self.basis_params(p, t)

        d = periodic_distance_torch(x, xi)
        phi = torch.exp(-(d ** 2) / (2.0 * h ** 2))
        u = torch.sum((gate * alpha) * phi, dim=-1)
        return u, (gate, alpha, xi, h)


# ============================================================
# Sampling
# ============================================================
def sample_p_batch(B: int, device: torch.device, x0_range=(0.2, 0.8), nu_range=(0.03, 0.12)) -> torch.Tensor:
    x0 = x0_range[0] + (x0_range[1] - x0_range[0]) * torch.rand(B, 1, device=device)

    u = torch.rand(B, 1, device=device)
    log_nu_min = math.log10(nu_range[0])
    log_nu_max = math.log10(nu_range[1])
    nu = 10.0 ** (log_nu_min + (log_nu_max - log_nu_min) * u)
    return torch.cat([x0, nu], dim=1)


def sample_interior_points_batch(B: int, N: int, device: torch.device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.rand(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


def sample_ic_points_batch(B: int, N: int, device: torch.device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.zeros(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


def sample_periodic_points_batch(B: int, N: int, device: torch.device):
    t = torch.rand(B, N, 1, device=device)
    xL = torch.zeros(B, N, 1, device=device)
    xR = torch.ones(B, N, 1, device=device)
    return torch.cat([xL, t], dim=-1), torch.cat([xR, t], dim=-1)


def sample_near_ic_points_batch(B: int, N: int, device: torch.device, t_max: float = 0.15):
    x = torch.rand(B, N, 1, device=device)
    t = t_max * torch.rand(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


# ============================================================
# PDE residual
# ============================================================
def advection_residual_batch(model: nn.Module, p: torch.Tensor, xt_int: torch.Tensor) -> torch.Tensor:
    xt_int = xt_int.clone().detach().requires_grad_(True)
    u, _ = model(p, xt_int)
    grads = torch.autograd.grad(
        outputs=u,
        inputs=xt_int,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]
    u_x = grads[:, :, 0]
    u_t = grads[:, :, 1]
    return u_t + u_x


# ============================================================
# Evaluation / plotting
# ============================================================
def exact_solution(x, t, x0, nu):
    s = np.remainder(x - t, 1.0)
    return periodic_gaussian_numpy(s, x0, nu)


def plot_training_history(history, save_path):
    plt.figure(figsize=(7, 4.5))
    for key in ["loss", "loss_pde", "loss_ic", "loss_per", "loss_center", "loss_width"]:
        plt.plot(history["epoch"], history[key], lw=1.8, label=key.replace("loss_", "").upper())
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training history")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=250)
    plt.close()


def evaluate_case_on_grid(model, device, x0_test, nu_test, Nx=200, Nt=200):
    x = np.linspace(0.0, 1.0, Nx, endpoint=False)
    t = np.linspace(0.0, 1.0, Nt)
    X, T = np.meshgrid(x, t, indexing="ij")

    xt_grid = np.stack([X.ravel(), T.ravel()], axis=1)
    xt_grid_t = torch.tensor(xt_grid, dtype=torch.float32, device=device).view(1, -1, 2)
    p_test = torch.tensor([[x0_test, nu_test]], dtype=torch.float32, device=device)

    model.eval()
    with torch.no_grad():
        u_pred_flat, _ = model(p_test, xt_grid_t)
    u_pred = u_pred_flat.cpu().numpy().reshape(Nx, Nt)

    u_exact = exact_solution(X, T, x0_test, nu_test)
    abs_err = np.abs(u_pred - u_exact)
    rel_l2 = np.linalg.norm((u_pred - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)
    return {
        "u_exact": u_exact,
        "u_pred": u_pred,
        "abs_err": abs_err,
        "rel_l2": rel_l2,
    }


def plot_multicase_fields(model, device, test_cases, save_dir, Nx=200, Nt=200):
    num_tests = len(test_cases)
    fig, axs = plt.subplots(3, num_tests, figsize=(4.4 * num_tests, 9.4), constrained_layout=True)
    if num_tests == 1:
        axs = axs.reshape(3, 1)

    results = []
    exacts, preds, errs = [], [], []
    for name, x0, nu in test_cases:
        out = evaluate_case_on_grid(model, device, x0, nu, Nx=Nx, Nt=Nt)
        exacts.append((name, x0, nu, out["u_exact"]))
        preds.append((name, x0, nu, out["u_pred"]))
        errs.append((name, x0, nu, out["abs_err"]))
        results.append({"name": name, "x0": x0, "nu": nu, "rel_l2": out["rel_l2"]})

    sol_vals = np.concatenate([u.ravel() for *_, u in exacts + preds])
    err_vals = np.concatenate([u.ravel() for *_, u in errs])
    sol_vmin, sol_vmax = float(sol_vals.min()), float(sol_vals.max())
    err_vmax = float(err_vals.max())
    im_sol = None
    im_err = None

    for j in range(num_tests):
        name, x0, nu, ue = exacts[j]
        _, _, _, up = preds[j]
        _, _, _, er = errs[j]

        ax = axs[0, j]
        im_sol = ax.imshow(ue.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        ax.set_title(rf"$(x_0,\nu)=({x0:.2f},{nu:.2f})$")
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Exact")

        ax = axs[1, j]
        ax.imshow(up.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Prediction")

        ax = axs[2, j]
        im_err = ax.imshow(er.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0.0, vmax=err_vmax)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$|u_{pred}-u_{exact}|$")

    cbar1 = fig.colorbar(im_sol, ax=axs[0:2, :], shrink=0.95, location="right")
    cbar1.set_label("Solution value")
    cbar2 = fig.colorbar(im_err, ax=axs[2, :], shrink=0.95, location="right")
    cbar2.set_label("Absolute error")

    out_path = os.path.join(save_dir, "multicase_fields.png")
    fig.savefig(out_path, dpi=250, bbox_inches="tight")
    plt.close(fig)

    save_standardized_bundle(test_cases, exacts, preds, errs, save_dir, Nx=Nx, Nt=Nt)
    return results



def save_standardized_bundle(test_cases, exacts, preds, errs, save_dir, Nx=200, Nt=200):
    """
    Save a standardized bundle for unified comparison plotting across methods.

    Required keys:
        x, t, params, case_names, u_exact, u_pred, abs_error
    """
    os.makedirs(save_dir, exist_ok=True)

    x = np.linspace(0.0, 1.0, Nx, endpoint=False).astype(np.float32)
    t = np.linspace(0.0, 1.0, Nt).astype(np.float32)

    case_names = np.asarray([name for name, _, _, _ in exacts])
    params = np.asarray([[x0, nu] for _, x0, nu, _ in exacts], dtype=np.float32)
    u_exact = np.stack([u for _, _, _, u in exacts], axis=0).astype(np.float32)
    u_pred = np.stack([u for _, _, _, u in preds], axis=0).astype(np.float32)
    abs_error = np.stack([u for _, _, _, u in errs], axis=0).astype(np.float32)

    bundle_path = os.path.join(save_dir, "kapi_bundle.npz")
    np.savez_compressed(
        bundle_path,
        x=x,
        t=t,
        params=params,
        case_names=case_names,
        u_exact=u_exact,
        u_pred=u_pred,
        abs_error=abs_error,
        method=np.asarray("KAPI"),
    )
    print(f"Saved standardized bundle: {bundle_path}")


def save_summary_csv(summary, save_dir):
    path = os.path.join(save_dir, "test_case_summary.csv")
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "x0", "nu", "rel_l2"])
        writer.writeheader()
        for row in summary:
            writer.writerow(row)


def visualize_dynamic_centers(model, device, test_cases, save_dir, K=60):
    model.eval()
    t_vals = np.linspace(0.0, 1.0, K)
    for name, x0, nu in test_cases:
        p = torch.tensor([[x0, nu]], dtype=torch.float32, device=device)
        t = torch.tensor(t_vals, dtype=torch.float32, device=device).view(1, K, 1)
        with torch.no_grad():
            alpha, xi, h, gate = model.basis_params(p, t)
        xi_np = xi[0].cpu().numpy()      # (K,M)
        h_np = h[0].cpu().numpy()
        w_np = np.abs((alpha[0] * gate[0]).cpu().numpy())

        X_pts = xi_np.reshape(-1)
        T_pts = np.repeat(t_vals, model.M)
        S_pts = np.clip(120.0 * h_np.reshape(-1), 8.0, 80.0)
        C_pts = w_np.reshape(-1)

        fig, ax = plt.subplots(figsize=(5.5, 5.0))
        sc = ax.scatter(X_pts, T_pts, c=C_pts, s=S_pts, cmap="viridis", alpha=0.85)
        plt.colorbar(sc, ax=ax, label=r"$|\alpha_j g_j|$")
        ax.scatter([x0], [0.0], color="red", marker="x", s=90, label="IC center")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel("x")
        ax.set_ylabel("t")
        ax.set_title(f"Dynamic basis centers: {name}")
        ax.legend(loc="upper right")
        fig.tight_layout()
        fig.savefig(os.path.join(save_dir, f"dynamic_centers_{name}.png"), dpi=250)
        plt.close(fig)


# ============================================================
# Training
# ============================================================
@dataclass
class TrainConfig:
    B_pde: int = 4
    N_int: int = 256
    N_ic: int = 128
    N_per: int = 128
    N_near: int = 96
    epochs: int = 5000
    lr: float = 1e-3
    print_every: int = 100
    nu_min: float = 0.03
    nu_max: float = 0.12
    nu_easy: float = 0.06
    lambda_ic: float = 10.0
    lambda_per: float = 1.0
    lambda_center: float = 5e-4
    lambda_width: float = 2e-4


def train_meta_advection(model: nn.Module, device: torch.device, cfg: TrainConfig):
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    best_loss = float("inf")
    best_state = None

    history = {k: [] for k in ["epoch", "loss", "loss_pde", "loss_ic", "loss_per", "loss_center", "loss_width"]}

    for ep in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad()

        if ep <= cfg.epochs // 2:
            nu_min_curr, nu_max_curr = cfg.nu_easy, cfg.nu_max
        else:
            nu_min_curr, nu_max_curr = cfg.nu_min, cfg.nu_max

        p_batch = sample_p_batch(cfg.B_pde, device, nu_range=(nu_min_curr, nu_max_curr))
        xt_int = sample_interior_points_batch(cfg.B_pde, cfg.N_int, device)
        xt_ic = sample_ic_points_batch(cfg.B_pde, cfg.N_ic, device)
        xtL, xtR = sample_periodic_points_batch(cfg.B_pde, cfg.N_per, device)
        xt_near = sample_near_ic_points_batch(cfg.B_pde, cfg.N_near, device, t_max=0.15)

        # PDE residual, with extra early-time points for stability.
        res_int = advection_residual_batch(model, p_batch, xt_int)
        res_near = advection_residual_batch(model, p_batch, xt_near)
        loss_pde = torch.mean(res_int ** 2) + 0.5 * torch.mean(res_near ** 2)

        # Exact initial condition fit.
        u_ic, _ = model(p_batch, xt_ic)
        x_ic = xt_ic[:, :, 0]
        x0 = p_batch[:, 0:1].expand_as(x_ic)
        nu = p_batch[:, 1:2].expand_as(x_ic)
        u_ic_true = periodic_gaussian_torch(x_ic, x0, nu)
        loss_ic = torch.mean((u_ic - u_ic_true) ** 2)

        # Periodic boundary.
        uL, _ = model(p_batch, xtL)
        uR, _ = model(p_batch, xtR)
        loss_per = torch.mean((uL - uR) ** 2)

        # Mild regularization to avoid chaotic particle motion/width collapse.
        _, (gate_aux, alpha_aux, xi_aux, h_aux) = model(p_batch, xt_near)
        dxi_dt = xi_aux[:, 1:, :] - xi_aux[:, :-1, :] if xi_aux.shape[1] > 1 else xi_aux * 0.0
        loss_center = torch.mean(torch.abs(dxi_dt)) + 0.2 * torch.mean(torch.abs(gate_aux * alpha_aux))
        target_h = p_batch[:, 1:2].unsqueeze(1)  # roughly align widths with IC scale
        loss_width = torch.mean((h_aux - torch.clamp(1.25 * target_h, model.h_min, model.h_max)) ** 2)

        loss = (
            loss_pde
            + cfg.lambda_ic * loss_ic
            + cfg.lambda_per * loss_per
            + cfg.lambda_center * loss_center
            + cfg.lambda_width * loss_width
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        history["epoch"].append(ep)
        history["loss"].append(loss.item())
        history["loss_pde"].append(loss_pde.item())
        history["loss_ic"].append(loss_ic.item())
        history["loss_per"].append(loss_per.item())
        history["loss_center"].append(loss_center.item())
        history["loss_width"].append(loss_width.item())

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % cfg.print_every == 0 or ep == 1:
            print(
                f"Epoch {ep:4d} | Loss {loss.item():.3e} | "
                f"PDE {loss_pde.item():.3e} | IC {loss_ic.item():.3e} | "
                f"PER {loss_per.item():.3e}"
            )

        if device.type == "cuda":
            torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


# ============================================================
# Main
# ============================================================
def main():
    parser = argparse.ArgumentParser(description="Dynamic-basis meta-SPINN without explicit characteristics")
    parser.add_argument("--gpu", action="store_true", help="Use GPU if available")
    parser.add_argument("--epochs", type=int, default=5000)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--M", type=int, default=32)
    parser.add_argument("--outdir", type=str, default="outputs_kapi")
    parser.add_argument("--model_path", type=str, default="kapi_advection.pt")
    args, _unknown = parser.parse_known_args()

    os.makedirs(args.outdir, exist_ok=True)
    device = get_device(args.gpu)
    print("Using device:", device)

    cfg = TrainConfig(epochs=args.epochs, lr=args.lr)
    model = DynamicMetaSPINN(M=args.M).to(device)
    model, history = train_meta_advection(model, device, cfg)

    model_path = os.path.join(args.outdir, args.model_path)
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to: {model_path}")

    plot_training_history(history, os.path.join(args.outdir, "training_loss.png"))

    loaded_model = DynamicMetaSPINN(M=args.M).to(device)
    loaded_model.load_state_dict(torch.load(model_path, map_location=device))
    loaded_model.eval()

    test_cases = [
        ("Center_in_range", 0.50, 0.07),
        ("Offcenter_in_range", 0.30, 0.09),
        ("Narrow_in_range", 0.65, 0.04),
        ("Narrow_out_range", 0.50, 0.02),
    ]
    summary = plot_multicase_fields(loaded_model, device, test_cases, args.outdir, Nx=200, Nt=200)
    save_summary_csv(summary, args.outdir)
    visualize_dynamic_centers(loaded_model, device, test_cases, args.outdir, K=60)

    print("\nTest summary:")
    for row in summary:
        print(f"{row['name']:20s} | x0={row['x0']:.3f}, nu={row['nu']:.3f} | relL2={row['rel_l2']:.3e}")
    print(f"\nAll outputs saved in: {args.outdir}")


if __name__ == "__main__":
    main()

Using device: cpu
Epoch    1 | Loss 6.045e+00 | PDE 4.232e+00 | IC 1.813e-01 | PER 2.125e-18
Epoch  100 | Loss 4.999e-01 | PDE 2.150e-01 | IC 2.849e-02 | PER 6.103e-17
Epoch  200 | Loss 2.691e-01 | PDE 1.733e-01 | IC 9.573e-03 | PER 3.135e-16
Epoch  300 | Loss 1.285e-01 | PDE 1.014e-01 | IC 2.702e-03 | PER 6.324e-16
Epoch  400 | Loss 1.124e-01 | PDE 8.960e-02 | IC 2.282e-03 | PER 5.696e-16
Epoch  500 | Loss 8.354e-02 | PDE 6.591e-02 | IC 1.761e-03 | PER 1.017e-15
Epoch  600 | Loss 5.594e-02 | PDE 4.204e-02 | IC 1.387e-03 | PER 1.026e-15
Epoch  700 | Loss 4.860e-02 | PDE 2.860e-02 | IC 1.998e-03 | PER 9.860e-16
Epoch  800 | Loss 8.857e-02 | PDE 7.985e-02 | IC 8.691e-04 | PER 3.105e-15
Epoch  900 | Loss 4.522e-02 | PDE 4.225e-02 | IC 2.952e-04 | PER 3.384e-15
Epoch 1000 | Loss 1.607e-02 | PDE 1.159e-02 | IC 4.452e-04 | PER 1.914e-15
Epoch 1100 | Loss 5.483e-02 | PDE 3.488e-02 | IC 1.992e-03 | PER 3.153e-16
Epoch 1200 | Loss 2.254e-02 | PDE 1.892e-02 | IC 3.596e-04 | PER 3.030e-15
Epoch 1

/tmp/ipykernel_30043/1468629754.py:564: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load(model_path, map_location=device))


Saved standardized bundle: outputs_kapi/kapi_bundle.npz

Test summary:
Center_in_range      | x0=0.500, nu=0.070 | relL2=4.022e-02
Offcenter_in_range   | x0=0.300, nu=0.090 | relL2=2.215e-02
Narrow_in_range      | x0=0.650, nu=0.040 | relL2=1.166e-01
Narrow_out_range     | x0=0.500, nu=0.020 | relL2=3.656e-01

All outputs saved in: outputs_kapi
